# Ministral judge — pinned GPU smoke test and evaluation

Uses a separate, non-Qwen model to score faithfulness, relevance, coherence and fluency.
Defaults to **two synthetic cases**, 20 samples per case. Smoke scores are not research results.

Select **Runtime → Change runtime type → A100 GPU**. The BF16 model download is large;
allow roughly 35 GB of free runtime disk for weights and the dedicated serving environment.
This notebook has been syntax-checked locally; the real GPU run is still required.


In [ ]:
from pathlib import Path
import datetime, json, os, subprocess, sys, time

RUN_MODE = "smoke"  # Leave this unchanged for the first run.
SAVE_TO_DRIVE = True
START_LOCAL_SERVER = True
BASE_URL = "http://127.0.0.1:8001/v1"
# Only for RUN_MODE="evaluation": 50 explanations + 50 profiles in the documented JSONL format.
REAL_CASES = None

REPO_URL = "https://github.com/Ashley96yy/data298a-team5-topic22.git"
CODE_COMMIT = "7ef50d3e9f0fc9a13dda2bc248bffb22e67318bd"
PROJECT = Path("/content/298A_ministral_judge")
ENV = Path("/content/ministral_judge_env")
PYTHON = ENV / "bin/python"
assert RUN_MODE in ("smoke", "evaluation")


## 1. Retrieve the exact committed implementation
The repository must be accessible to your GitHub account. If it is private, authenticate
with an approved method before this cell; never paste credentials into the notebook.


In [ ]:
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--branch", "abhinita/ministral-judge", "--single-branch", REPO_URL, str(PROJECT)], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", CODE_COMMIT], check=True)
head = subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
assert head == CODE_COMMIT, "Existing checkout differs; use a fresh PROJECT directory."
print("Code commit:", head)
print("Judge:", json.loads((PROJECT / "configs/judge/ministral.json").read_text())["model_id"])


## 2. Create an isolated serving environment
This avoids replacing the notebook runtime's PyTorch/Transformers installation.
Installation and the first model download may take several minutes.


In [ ]:
if not PYTHON.exists():
    subprocess.run([sys.executable, "-m", "venv", str(ENV)], check=True)
if START_LOCAL_SERVER:
    subprocess.run([str(PYTHON), "-m", "pip", "install", "-r", str(PROJECT / "src/evaluation/judge/requirements-serving.txt")], check=True)
    subprocess.run([str(PYTHON), "-c", "import torch; print('GPU:', torch.cuda.get_device_name(0)); print('VRAM GiB:', torch.cuda.get_device_properties(0).total_memory/2**30); assert torch.cuda.is_bf16_supported()"], check=True)
subprocess.run([str(PYTHON), "-m", "unittest", "discover", "-s", "tests", "-p", "test_judge.py", "-v"], cwd=PROJECT, check=True)


## 3. Choose a persistent output folder
Each execution uses a new timestamped folder. The committed synthetic cases require
no dataset files. Real input and output files should stay in controlled storage.


In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/298A/geval_judge_runs")
else:
    OUTPUT_ROOT = Path("/content/geval_judge_runs")
RUN_ID = RUN_MODE + "_" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR = OUTPUT_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
CASES = PROJECT / "tests/fixtures/judge/synthetic_cases.jsonl" if RUN_MODE == "smoke" else Path(REAL_CASES or "MISSING_REAL_CASES")
assert CASES.is_file(), "Set REAL_CASES to the frozen evaluation JSONL."
print("Outputs:", RUN_DIR)


## 4. Start the judge and wait for readiness
This starts a localhost-only API. It does not deploy a shared cloud service.
For an existing approved endpoint, set START_LOCAL_SERVER=False and its HTTPS BASE_URL.


In [ ]:
server = None
log = None
if START_LOCAL_SERVER:
    log = (RUN_DIR / "server.log").open("w")
    server = subprocess.Popen([str(PYTHON), "-m", "src.evaluation.judge.serve", "--record-dir", str(RUN_DIR / "serving")], cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    print("Starting judge; progress is in", RUN_DIR / "server.log")
else:
    from getpass import getpass
    key = getpass("Endpoint API key (leave empty if not required): ")
    if key: os.environ["JUDGE_API_KEY"] = key

check_code = "from src.evaluation.judge.run import Endpoint,load_config; import os; Endpoint(os.environ['JUDGE_BASE_URL'],os.environ.get('JUDGE_API_KEY'),timeout=10).verify(load_config())"
check_env = {**os.environ, "JUDGE_BASE_URL": BASE_URL}
deadline = time.monotonic() + 1200
while time.monotonic() < deadline:
    if server is not None and server.poll() is not None:
        raise RuntimeError("Server exited. Inspect server.log; do not proceed to scoring.")
    ready = subprocess.run([str(PYTHON), "-c", check_code], cwd=PROJECT, env=check_env, capture_output=True)
    if ready.returncode == 0:
        print("Ready: expected pinned judge alias is available.")
        break
    time.sleep(10)
else:
    raise TimeoutError("Judge not ready after 20 minutes. Inspect server.log.")


## 5. Score the cases
Every case needs all 20 valid score samples. A failed or partial run must not be
reported as a final result. Human agreement is a separate required evaluation.


In [ ]:
subprocess.run([str(PYTHON), "-m", "src.evaluation.judge.run", "--cases", str(CASES), "--output", str(RUN_DIR / "scores"), "--mode", RUN_MODE, "--base-url", BASE_URL], cwd=PROJECT, check=True)
metadata = json.loads((RUN_DIR / "scores/run_metadata.json").read_text())
print(json.dumps({k: metadata[k] for k in ["status", "mode", "cases_completed", "cases_expected", "seconds", "calibration_status"]}, indent=2))
print("Keep this folder:", RUN_DIR)


## 6. Stop the local judge when finished
Run this cell even if scoring failed, then disconnect the GPU runtime when done.


In [ ]:
import signal
if server is not None and server.poll() is None:
    os.killpg(server.pid, signal.SIGTERM)
    try:
        server.wait(timeout=30)
    except subprocess.TimeoutExpired:
        os.killpg(server.pid, signal.SIGKILL)
        server.wait(timeout=10)
if log is not None:
    log.close()
print("Local server stopped. Saved outputs:", RUN_DIR)
